# Deep Learning Quiz

**Name:** Imama Tanveer  
**Roll No:** 2023-SE-28  
**Topic:** Preprocessing – Border Removal and Slip Segmentation

## 1. Border Removal

The input image contains an unwanted frame around the scanned slips. The first step removes that frame by cropping fixed margins from the four sides. This makes the later contour detection more reliable.

In [ ]:
from pathlib import Path
import cv2

PROJECT_DIR = Path.cwd()
IMAGE_FILE = PROJECT_DIR / "input_image.jpeg"
CROPPED_FILE = PROJECT_DIR / "preprocessed_input_image.jpg"

image = cv2.imread(str(IMAGE_FILE))

if image is None:
    raise FileNotFoundError(f"Input image was not found: {IMAGE_FILE}")

height, width = image.shape[:2]

# Margins used to remove the scanner frame
crop_top = 22
crop_bottom = 33
crop_left = 26
crop_right = 16

cropped = image[crop_top:height-crop_bottom,
                crop_left:width-crop_right]

cv2.imwrite(str(CROPPED_FILE), cropped)

print(f"Original size : {width} x {height}")
print(f"Cropped size  : {cropped.shape[1]} x {cropped.shape[0]}")
print(f"Saved to      : {CROPPED_FILE.name}")


## 2. Recursive Sobel Splitting

Some contours may contain more than one receipt joined vertically. These large regions are treated as mega-slips. A horizontal Sobel gradient is used to locate a low-edge area near the middle, and the region is split recursively until individual slips remain.

In [ ]:
import numpy as np

def split_mega_slip(image, box, save_dir, marked_image, counter):
    x, y, w, h = box
    region = image[y:y+h, x:x+w]

    gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
    vertical_gradient = cv2.Sobel(
        gray_region, cv2.CV_64F, 0, 1, ksize=3
    )
    gradient_strength = np.abs(vertical_gradient).astype(np.uint8)
    row_score = np.sum(gradient_strength, axis=1)

    # Try a split only for regions large enough to hold multiple slips.
    if h > 1050:
        start = int(h * 0.30)
        end = int(h * 0.70)
        split_at = start + int(np.argmin(row_score[start:end]))

        if row_score[split_at] < np.mean(row_score) * 0.5:
            counter = split_mega_slip(
                image, (x, y, w, split_at), save_dir,
                marked_image, counter
            )
            counter = split_mega_slip(
                image, (x, y + split_at, w, h - split_at),
                save_dir, marked_image, counter
            )
            return counter

    filename = save_dir / f"slip_{counter}.jpg"
    cv2.imwrite(str(filename), region)

    cv2.rectangle(marked_image, (x, y), (x+w, y+h), (255, 255, 0), 4)
    cv2.putText(
        marked_image, f"Split {counter}: {w}x{h}",
        (x + 5, y + 30), cv2.FONT_HERSHEY_SIMPLEX,
        0.7, (255, 255, 0), 2
    )

    print(f"Saved split slip {counter}: {w}x{h}")
    return counter + 1


## 3. Slip Detection and Segmentation

The main pipeline converts the image to grayscale, reduces noise with Gaussian blur, detects edges with Canny, and connects useful regions using morphological operations. External contours are then checked against the required size ranges. Normal slips are saved directly, while taller mega-slips are passed to the recursive Sobel routine.

In [ ]:
def detect_and_save_slips(image_file, output_folder):
    image = cv2.imread(str(image_file))
    if image is None:
        raise FileNotFoundError(f"Could not read: {image_file}")

    output_folder.mkdir(parents=True, exist_ok=True)
    marked = image.copy()

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    smooth = cv2.GaussianBlur(gray, (9, 9), 0)
    edges = cv2.Canny(smooth, 30, 150)

    close_kernel = np.ones((120, 1), np.uint8)
    connected = cv2.morphologyEx(
        edges, cv2.MORPH_CLOSE, close_kernel
    )
    expanded = cv2.dilate(
        connected, np.ones((3, 3), np.uint8), iterations=2
    )

    contours, _ = cv2.findContours(
        expanded, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    slip_number = 1

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)

        # Regular slip dimensions
        regular_slip = 250 <= w <= 500 and 500 <= h <= 1100

        if regular_slip:
            slip = image[y:y+h, x:x+w]
            cv2.imwrite(str(output_folder / f"slip_{slip_number}.jpg"), slip)

            cv2.rectangle(marked, (x, y), (x+w, y+h), (0, 255, 0), 3)
            cv2.putText(
                marked, f"Slip {slip_number}: {w}x{h}",
                (x, max(20, y-10)), cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (0, 255, 0), 2
            )

            print(f"Found regular slip {slip_number}: {w}x{h}")
            slip_number += 1

        elif h > 1100:
            print(f"Mega-slip found: {w}x{h}")
            slip_number = split_mega_slip(
                image, (x, y, w, h), output_folder,
                marked, slip_number
            )

    cv2.imwrite(str(PROJECT_DIR / "annotated_evaluation.jpg"), marked)
    return len(list(output_folder.glob("slip_*.jpg")))


## 4. Run the Complete Process

The cropped image generated in the first section is used as the input for segmentation. The detected slips are stored in `output_slips`, and the final annotated image is saved in the project folder.

In [ ]:
OUTPUT_DIR = PROJECT_DIR / "output_slips"

# Remove old generated slip files before running again.
for old_file in OUTPUT_DIR.glob("slip_*.jpg"):
    old_file.unlink()

total = detect_and_save_slips(CROPPED_FILE, OUTPUT_DIR)

print(f"\nProcessing complete. {total} slip(s) saved.")
print("Check output_slips and annotated_evaluation.jpg for the results.")
